# NYC HANES HCV benchmark

Run from the project root or `analysis/`. The default uses saved responses and writes `nyc_hanes_mad.csv` and `nyc_hanes_auc.csv` in `analysis_results/`.


## Optional profile generation

Set `BUILD_PROFILES=True` to read the six ICPSR Stata files and construct the 35-positive/35-negative cohort. Model generation requires both `BUILD_PROFILES=True` and `RUN_GENERATION=True`.

HCV labels 1 and 2 mean antibody-positive and antibody-negative, respectively. The antibody label is excluded from model prompts.


In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((candidate for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent)
             if (candidate / "analysis" / "analysis_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the project root or its analysis folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import numpy as np
import json
from analysis.analysis_utils import roc_auc_score, spearmanr, DATA_DIR, RESULT_DIR, MODELS

# Saved-response analysis is the default
RUN_GENERATION = False
BUILD_PROFILES = False
if RUN_GENERATION and not BUILD_PROFILES:
    raise ValueError("Model generation requires BUILD_PROFILES=True to construct the NYC HANES prompts.")
DATA_ROOT = ROOT / "ICPSR_31421"
OUTPUT_JSONL = DATA_DIR / "outputs_nyc_hanes.jsonl"
OUTPUT_CSV = DATA_DIR / "outputs_nyc_hanes.csv"
B = 1000
SEED = 42

In [ ]:
if BUILD_PROFILES:
    file_paths = {
        'spfile': 'DS0001/31421-0001-Data.dta',   
        'capi':   'DS0002/31421-0002-Data.dta',   
        'acasi':  'DS0003/31421-0003-Data.dta',   
        'cidi':   'DS0004/31421-0004-Data.dta',  
        'exam':   'DS0005/31421-0005-Data.dta', 
        'lab':    'DS0006/31421-0006-Data.dta',}

    dfs = {}
    for name, path in file_paths.items():
        path = DATA_ROOT / path
        if not path.exists():
            raise FileNotFoundError(f'Missing NYC HANES file: {path}')
        dfs[name] = pd.read_stata(path, convert_categoricals=False)
        print(f"{name:8s}: {dfs[name].shape[0]} rows, {dfs[name].shape[1]} cols")

    # Merge on participant ID and keep the first copy of repeated columns
    df_all = dfs['spfile'].copy()
    for name in ['capi', 'acasi', 'cidi', 'exam', 'lab']:
        df = dfs[name]
        new_cols = ['SP_ID'] + [c for c in df.columns if c not in df_all.columns]
        df_all = pd.merge(df_all, df[new_cols], on='SP_ID', how='outer', validate='one_to_one')

    print(f"\nMerged dataset: {df_all.shape[0]} rows, {df_all.shape[1]} columns")
else:
    print("NYC HANES profile construction skipped; using saved responses.")


In [ ]:
if BUILD_PROFILES:
    required_vars = ["RIAGENDR","RIAAGEYR","RACE_ETH","DMQ140","HIQ012","HUQ010","BMXBMI","BMXWAIST","BPXSAR","BPXDAR","LBXGH","LBXGLU","LBXTC","LBXHDD","LBXBPB","LBXBCD","LBXTHG","FSTHOURS"]
    RANDOM_STATE, SAMPLE_PER_GROUP = 42, 35
    hcv_pos_all = df_all.loc[df_all["HCV"].eq(1)].copy()
    hcv_neg_all = df_all.loc[df_all["HCV"].eq(2)].copy()
    print("Missing required fields among all HCV-positive records:")
    print(hcv_pos_all[required_vars].isna().sum().loc[lambda x:x.gt(0)])
    if len(hcv_pos_all) != SAMPLE_PER_GROUP:
        raise ValueError(f"Only {len(hcv_pos_all)} HCV-positive records found; paper requires 35.")
    # Keep all 35 antibody-positive participants
    pos_pool = hcv_pos_all
    # Require these 18 complete fields for antibody-negative participants
    neg_pool = hcv_neg_all.dropna(subset=required_vars)
    if len(neg_pool) < SAMPLE_PER_GROUP:
        raise ValueError(f"Complete HCV-negative pool too small: HCV-={len(neg_pool)}; paper requires 35.")
    sample_pos = pos_pool.sample(SAMPLE_PER_GROUP, random_state=RANDOM_STATE)
    sample_neg = neg_pool.sample(SAMPLE_PER_GROUP, random_state=RANDOM_STATE)
    sample_df = pd.concat([sample_pos, sample_neg]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    assert sample_df.HCV.value_counts().to_dict() == {1:35, 2:35}
    print(f"Final sample size: {len(sample_df)} (35 HCV+, 35 HCV-)")

In [ ]:
if BUILD_PROFILES:
    gender_map = {
        1: "Male",
        2: "Female"}

    race_eth_map = {
        1: "Non-Hispanic White",
        2: "Non-Hispanic Black",
        3: "Non-Hispanic Asian",
        4: "Hispanic",
        5: "Non-Hispanic Other/Multi-racial"}

    education_map = {
        0: "never attended school or kindergarten only",
        1: "completed 1st grade",
        2: "completed 2nd grade",
        3: "completed 3rd grade",
        4: "completed 4th grade",
        5: "completed 5th grade",
        6: "completed 6th grade",
        7: "completed 7th grade",
        8: "completed 8th grade",
        9: "completed 9th grade",
        10: "completed 10th grade",
        11: "completed 11th grade",
        12: "completed 12th grade, no diploma",
        13: "high school graduate",
        14: "GED or equivalent",
        15: "some college, no degree",
        16: "associate degree (occupational/technical/vocational)",
        17: "associate degree (academic program)",
        18: "bachelor's degree",
        19: "master's degree",
        20: "professional school degree (e.g., MD, JD)",
        21: "doctoral degree (e.g., PhD, EdD)",
        99: "unknown education level"}

    insurance_map = {
        1: "has health insurance coverage",
        2: "does not have health insurance coverage",
        9: "unknown insurance status"}

    health_status_map = {
        1: "excellent",
        2: "very good",
        3: "good",
        4: "fair",
        5: "poor"}

    smoking_map = {
        1: "smokes cigarettes every day",
        2: "smokes cigarettes some days",
        3: "does not currently smoke cigarettes"}

    yes_no_map = {
        1: "yes",
        2: "no",
        7: "refused to answer",
        9: "does not know"}

    def safe_label(value, mapping, default="Not reported"):
        if pd.isna(value):
            return default
        return mapping.get(value, default)

    def safe_num(value, fmt="{:.1f}"):
        if pd.isna(value):
            return "Not reported"
        return fmt.format(value)

    prompt_list = []

    for idx, row in sample_df.iterrows():
        age = safe_num(row["RIAAGEYR"], "{:.0f}")
        sex = safe_label(row["RIAGENDR"], gender_map, "unknown sex")
        race = safe_label(
            row["RACE_ETH"],
            race_eth_map,
            "unspecified race/ethnicity")
        education = safe_label(
            row["DMQ140"],
            education_map,
            "unknown education level")
        insurance = safe_label(row["HIQ012"], insurance_map)
        health = safe_label(row["HUQ010"], health_status_map)
        smoking = safe_label(
            row["SMQ040"],
            smoking_map,
            "unknown smoking status")
        street_drugs = safe_label(
            row["DUQ100"],
            yes_no_map,
            "unknown")
        needle_use = safe_label(
            row["DUQ120"],
            yes_no_map,
            "unknown")

        bmi = safe_num(row["BMXBMI"], "{:.1f}")
        waist = safe_num(row["BMXWAIST"], "{:.1f}")
        sbp = safe_num(row["BPXSAR"], "{:.0f}")
        dbp = safe_num(row["BPXDAR"], "{:.0f}")
        hba1c = safe_num(row["LBXGH"], "{:.1f}")
        cholesterol = safe_num(row["LBXTC"], "{:.0f}")
        hdl = safe_num(row["LBXHDD"], "{:.0f}")

        prompt = (
            "This individual was a resident of New York City in 2004. "
            "The following individual characteristics were collected during that period.\n\n"

            "Individual profile:\n"
            f"- Age: {age} years\n"
            f"- Sex: {sex}\n"
            f"- Race/ethnicity: {race}\n"
            f"- Education: {education}\n"
            f"- Health insurance status: {insurance}\n"
            f"- Self-reported general health status: {health}\n"
            f"- Smoking status: {smoking}\n"
            f"- Ever used cocaine or other street drugs (not marijuana): "
            f"{street_drugs}\n"
            f"- Ever used a needle to take street drugs: "
            f"{needle_use}\n"
            f"- Body mass index (BMI): {bmi} kg/m²\n"
            f"- Waist circumference: {waist} cm\n"
            f"- Average systolic blood pressure: {sbp} mmHg\n"
            f"- Average diastolic blood pressure: {dbp} mmHg\n"
            f"- Hemoglobin A1c: {hba1c}%\n"
            f"- Total cholesterol: {cholesterol} mg/dL\n"
            f"- HDL cholesterol: {hdl} mg/dL\n\n"

            "Question:\n"
            "Estimate the individual's relative likelihood of HCV infection based on the "
            "provided characteristics.\n\n"

            "Output Format:\n"
            "Return ONLY a single-line JSON object. Do not include explanations, "
            "markdown, code blocks, or additional text.\n\n"

            "The JSON must exactly match the following format:\n"
            "{\n"
            '  "HCV_risk_score": integer,\n'
            '  "risk_factor_1": "variable_name",\n'
            '  "risk_factor_2": "variable_name",\n'
            '  "risk_factor_3": "variable_name"\n'
            "}\n\n"

            "Mapping:\n"
            "- HCV_risk_score: integer from 0 to 100, where 0 indicates the lowest "
            "expected likelihood of HCV infection and 100 indicates the highest "
            "expected likelihood of HCV infection.\n"
            "- risk_factor_1, risk_factor_2, risk_factor_3: the three characteristics "
            "from the provided profile that most influenced the estimated HCV risk. "
            "Return only one of the following variable names: "
            "age, sex, race_ethnicity, education, health_insurance, "
            "self_rated_health, smoking_status, street_drug_use, "
            "injection_drug_use, BMI, waist_circumference, blood_pressure, "
            "HbA1c, cholesterol, HDL. "
            "If fewer than three factors are relevant, return \"None\" for the remaining fields."
        )

        prompt_list.append({
            "SP_ID": row["SP_ID"],
            "true_HCV": row["HCV"],
            "prompt": prompt})

    print(prompt_list[0]["prompt"])

In [ ]:
if RUN_GENERATION:
    import boto3
    client = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
    
    model_id = "deepseek.v3.2"
    responses_list = []
    existing = set()
    if OUTPUT_CSV.is_file():
        saved = pd.read_csv(OUTPUT_CSV)
        existing.update(saved[["Model", "SP_ID"]].itertuples(index=False, name=None))
    if OUTPUT_JSONL.is_file():
        for line in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines():
            entry = json.loads(line)
            existing.add((entry["Model"], entry["SP_ID"]))
    
    for prompt_entry in prompt_list:
        current_id = prompt_entry["SP_ID"]
        if (model_id, current_id) in existing:
            continue
        api_text = prompt_entry["prompt"]
        person_row = sample_df[sample_df["SP_ID"] == current_id]
    
        if person_row.empty:
            continue
    
        row = person_row.iloc[0]
        messages = [{
            "role": "user",
            "content": [{"text": api_text}]}]
    
        response = client.converse(
            modelId=model_id,
            messages=messages,
            inferenceConfig={
                "temperature": 0})
    
        raw_message_text = ""
    
        try:
            content_blocks = (
                response
                .get("output", {})
                .get("message", {})
                .get("content", []))
            texts = []
            for block in content_blocks:
                if isinstance(block, dict) and "text" in block:
                    texts.append(block["text"])
            raw_message_text = " ".join(texts).strip()
        except Exception as e:
            print("Parsing error:", e)
            raw_message_text = ""
    
        try:
            structured_response = json.loads(raw_message_text)
        except json.JSONDecodeError:
            structured_response = raw_message_text
    
        entry = {
            "SP_ID": current_id,
            "Model": model_id,
            "true_HCV": row["HCV"],
    
            "Age": safe_num(row["RIAAGEYR"], "{:.0f}"),
            "Sex": safe_label(row["RIAGENDR"], gender_map, "unknown sex"),
            "Race_ethnicity": safe_label(row["RACE_ETH"], race_eth_map, "unspecified race/ethnicity"),
            "Education": safe_label(row["DMQ140"], education_map),
            "Health_insurance": safe_label(row["HIQ012"], insurance_map),
            "Self_rated_health": safe_label(row["HUQ010"], health_status_map),
            "Smoking_status": safe_label(row["SMQ040"], smoking_map, "unknown smoking status"),
            "Street_drug_use": safe_label(row["DUQ100"], yes_no_map, "unknown"),
            "Injection_drug_use": safe_label(row["DUQ120"], yes_no_map, "unknown"),
    
            "BMI": row["BMXBMI"],
            "Waist_circumference": row["BMXWAIST"],
            "Systolic_BP": row["BPXSAR"],
            "Diastolic_BP": row["BPXDAR"],
            "HbA1c": row["LBXGH"],
            "Total_cholesterol": row["LBXTC"],
            "HDL": row["LBXHDD"],
    
            "response": structured_response
        }
    
    
        OUTPUT_JSONL.parent.mkdir(exist_ok=True)
        with OUTPUT_JSONL.open("a", encoding="utf-8") as f:
            f.write(json.dumps(entry) + "\n")
        responses_list.append(entry)
        existing.add((model_id, current_id))
else:
    print("Generation skipped; analyzing saved NYC HANES HCV antibody benchmark outputs.")


In [ ]:
if RUN_GENERATION:
    rows = []
    for line in OUTPUT_JSONL.read_text(encoding="utf-8").splitlines():
        metadata = json.loads(line)
        response = metadata.pop("response")
        if isinstance(response, str):
            response = json.loads(response.replace("```json", "").replace("```", "").strip())
        if not isinstance(response, dict):
            raise ValueError("NYC HANES response must be a JSON object.")
        if any(k in response for k in ["SP_ID", "Model", "true_HCV"]):
            raise ValueError("Response must not overwrite profile identifiers or antibody status.")
        rows.append({**response, **metadata})
    generated_df = pd.DataFrame(rows)
    if generated_df.duplicated(["Model", "SP_ID"]).any():
        raise ValueError("Duplicate NYC HANES profile outputs.")
    generated_csv = OUTPUT_JSONL.with_suffix(".csv")
    generated_df.to_csv(generated_csv, index=False)
    OUTPUT_CSV = generated_csv


In [ ]:
df = pd.read_csv(OUTPUT_CSV)
continuous = ["Age", "BMI", "Waist_circumference", "Systolic_BP", "Diastolic_BP",
              "HbA1c", "Total_cholesterol", "HDL"]
profile_fields = ["true_HCV", "Age", "Sex", "Race_ethnicity", "Education",
                  "Health_insurance", "Self_rated_health", "Smoking_status",
                  "Street_drug_use", "Injection_drug_use"] + continuous[1:]
required = ["Model", "SP_ID", "HCV_risk_score"] + profile_fields
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f"Missing NYC HANES saved-output columns: {missing}")
if df[["Model", "SP_ID"]].isna().any().any() or df.duplicated(["Model", "SP_ID"]).any():
    raise ValueError("Missing or duplicate NYC HANES model/participant identifiers.")
if set(df.Model) != set(MODELS):
    raise ValueError("NYC HANES outputs must contain all three evaluated models.")
score = pd.to_numeric(df.HCV_risk_score, errors="raise")
if not np.isfinite(score).all() or not score.between(0, 100).all() or not score.eq(score.round()).all():
    raise ValueError("HCV scores must be finite integers from 0 to 100.")
df["HCV_risk_score"] = score
for column in continuous:
    df[column] = pd.to_numeric(df[column].replace("Not reported", np.nan), errors="raise")
    if np.isinf(df[column]).any():
        raise ValueError(f"Infinite NYC HANES covariate values: {column}")
df["HCV_binary"] = df["true_HCV"].map({1: 1, 2: 0})
if df.HCV_binary.isna().any():
    raise ValueError("Unexpected NYC HANES antibody-status labels.")

# Match participant order across models
reference = df.loc[df.Model.eq(MODELS[0])].set_index("SP_ID")
aligned_groups = []
for model in MODELS:
    group = df.loc[df.Model.eq(model)].set_index("SP_ID")
    if len(group) != 70 or set(group.index) != set(reference.index):
        raise ValueError(f"{model}: expected the same 70 NYC HANES participants.")
    if group.true_HCV.value_counts().to_dict() != {1: 35, 2: 35}:
        raise ValueError(f"{model}: expected 35 antibody-positive and 35 antibody-negative participants.")
    group = group.loc[reference.index]
    if not group[profile_fields].equals(reference[profile_fields]):
        raise ValueError(f"{model}: NYC HANES participant metadata differ between models.")
    aligned_groups.append(group.reset_index())
df = pd.concat(aligned_groups, ignore_index=True)

def spearman_pair(x, y):
    values = np.column_stack([np.asarray(x, float), np.asarray(y, float)])
    values = values[np.isfinite(values).all(axis=1)]
    if len(values) < 3 or np.unique(values[:, 0]).size < 2 or np.unique(values[:, 1]).size < 2:
        return np.nan
    return float(spearmanr(values[:, 0], values[:, 1]).statistic)

df["Sex_binary"] = df.Sex.astype(str).str.strip().str.lower().map({"female": 1, "male": 0})
df["Smoking_binary"] = df.Smoking_status.astype(str).str.strip().str.lower().map({
    "smokes cigarettes every day": 1, "smokes cigarettes some days": 1,
    "does not currently smoke cigarettes": 0})
df["Street_drug_binary"] = df.Street_drug_use.astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
df["Injection_drug_binary"] = df.Injection_drug_use.astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
binary = {"Sex": "Sex_binary", "Smoking_status": "Smoking_binary",
          "Street_drug_use": "Street_drug_binary", "Injection_drug_use": "Injection_drug_binary"}
all_vars = continuous + list(binary)

def nyc_hanes_metrics(group, seed):
    """Return AUC and the fixed 12-covariate MAD for matched participants."""
    if B < 1:
        raise ValueError("At least one NYC HANES bootstrap resample is required.")
    rng = np.random.default_rng(seed)
    positive = group.index[group.HCV_binary.eq(1)].to_numpy()
    negative = group.index[group.HCV_binary.eq(0)].to_numpy()
    if len(positive) != 35 or len(negative) != 35:
        raise ValueError("NYC HANES requires 35 participants per antibody-status stratum.")

    def one(positive_indices, negative_indices):
        selected = pd.concat([group.loc[positive_indices], group.loc[negative_indices]], axis=0)
        if len(positive_indices) != 35 or len(negative_indices) != 35 or not selected.HCV_binary.value_counts().eq(35).all():
            raise ValueError("Every NYC HANES replicate must retain 35 participants per HCV stratum.")
        y = selected.HCV_binary.to_numpy(float)
        scores = selected.HCV_risk_score.to_numpy(float)
        auc = roc_auc_score(y, scores)
        differences = []
        for variable in all_vars:
            column = variable if variable in continuous else binary[variable]
            x = selected[column].to_numpy(float)
            antibody_rho = spearman_pair(x, y)
            score_rho = spearman_pair(x, scores)
            differences.append(abs(antibody_rho - score_rho))
        differences = np.asarray(differences, float)
        n_covariates = int(np.isfinite(differences).sum())
        # MAD requires all 12 covariates in every resample
        mad = float(differences.mean()) if n_covariates == len(all_vars) else np.nan
        return auc, mad, n_covariates

    observed = one(positive, negative)
    if observed[2] != len(all_vars):
        raise ValueError("Observed NYC HANES MAD requires all 12 prespecified covariates.")
    draws = [one(rng.choice(positive, len(positive), replace=True),
                 rng.choice(negative, len(negative), replace=True)) for _ in range(B)]
    return observed, np.asarray(draws, float)

performance = []
nyc_hanes_results = []
for model, group in df.groupby("Model"):
    (auc, mad, n_covariates), draws = nyc_hanes_metrics(group.copy(), SEED)
    auc_valid = np.isfinite(draws[:, 0])
    mad_valid = np.isfinite(draws[:, 1])
    auc_ci = np.percentile(draws[:, 0], [2.5, 97.5]) if auc_valid.all() else [np.nan, np.nan]
    mad_ci = np.percentile(draws[:, 1], [2.5, 97.5]) if mad_valid.all() else [np.nan, np.nan]
    metadata = {"Model": model, "N": len(group), "Bootstrap_strata": "HCV antibody status",
                "Bootstrap_seed": SEED, "N_strata": 2, "N_participants_per_stratum": 35,
                "bootstrap_total": B}
    performance.append({**metadata, "AUC": auc, "AUC_lower_95": auc_ci[0], "AUC_upper_95": auc_ci[1],
                        "bootstrap_valid": int(auc_valid.sum()),
                        "ci_status": "ok" if auc_valid.all() else "undefined replicates; CI withheld"})
    nyc_hanes_results.append({**metadata, "NYC_HANES_MAD": mad, "MAD_lower_95": mad_ci[0],
                             "MAD_upper_95": mad_ci[1], "Bootstrap_B": B, "N_covariates": n_covariates,
                             "bootstrap_covariates_min": int(draws[:, 2].min()),
                             "bootstrap_covariates_max": int(draws[:, 2].max()),
                             "bootstrap_valid": int(mad_valid.sum()),
                             "ci_status": "ok" if mad_valid.all() else "undefined replicates; CI withheld"})
performance_df = pd.DataFrame(performance)
nyc_hanes_mad_df = pd.DataFrame(nyc_hanes_results)
print(performance_df.round(3).to_string(index=False))
print(nyc_hanes_mad_df.round(3).to_string(index=False))
performance_df.to_csv(RESULT_DIR / "nyc_hanes_auc.csv", index=False)
nyc_hanes_mad_df.to_csv(RESULT_DIR / "nyc_hanes_mad.csv", index=False)
